# Gold Layer 
--------------------------------------------
- Add Business logic to data
- To build the curated data for ML and insights


# Import necessarry libraries
--------------------------------

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import requests
from utill_funcs import get_winner
import plotly.express as px
from utill_funcs import add_home_or_away_winner
from pyspark.sql import Window

#Add Parameters
-------------------

In [0]:
catalog = 'wc_2026_predions'
gold_schema = 'gold'
silver_schema = 'silver'
gold_table_name = 'internation_results_data'
silver_table_name = 'cleaned_data'

#Read in the silver data
----------------------------

In [0]:
df_silver = spark.read.table(f'{catalog}.{silver_schema}.{silver_table_name}')
display(df_silver)

#Add Bi Logic
-----------------

In [0]:
### winner home or away team or draw
df_silver = add_home_or_away_winner(df_silver)

In [0]:
display(df_silver)

# Add New Features
---------------------------------
- Given that goals won't be known during new fixures it can't be a feature.
- We can add a rolling goals count up untill that fixure for each teams :
1. home_avg_goals_to_date
2. away_avg_goals_to_date

- Add recency 
- past_10_games result
- if win +1, draw 0 , loss -1
i..e England : WWLDW -> score of 2

In [0]:
# Add unique id to each entry
df_silver = df_silver.withColumn('id', monotonically_increasing_id())

In [0]:
### split dataframes 
df_home = df_silver.select('id','date','home_team','home_score')
df_away = df_silver.select('id','date','away_team','away_score')

# rename column to make them align
df_home = df_home.withColumnRenamed('home_team','team').withColumnRenamed('home_score','score')
df_away = df_away.withColumnRenamed('away_team','team').withColumnRenamed('away_score','score')
# combine dataframes
df = df_home.union(df_away)
display(df)

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

order_window = (
    Window
    .partitionBy("team")
    .orderBy("date")
)

previous_games_window = (
    Window
    .partitionBy("team")
    .orderBy("date")
    .rowsBetween(Window.unboundedPreceding, -1)
)

df = (
    df
    .withColumn(
        "game_number",
        F.row_number().over(order_window)
    )
    .withColumn(
        "avg_goals_before_match",
        F.coalesce(
            F.avg("score").over(previous_games_window),
            F.lit(0.0)
        )
    )
)

display(df)

In [0]:
### Now we require two joins whereby the home tema and away team get populated 
df = df.select('id','team','avg_goals_before_match','game_number')

In [0]:
from pyspark.sql import functions as F

s = df_silver.alias("s")
t = df.alias("t")

home_joined_df = (
    s.join(
        t,
        (F.col("s.id") == F.col("t.id")) &
        (F.col("s.home_team") == F.col("t.team")),
        "left"
    )
    .select(
        "s.*",
        F.col("t.avg_goals_before_match").alias("home_avg_goals"),
        F.col("t.game_number").alias("home_game_number")
    )
)

away_joined_df = (
    home_joined_df.alias("s")
    .join(
        t,
        (F.col("s.id") == F.col("t.id")) &
        (F.col("s.away_team") == F.col("t.team")),
        "left"
    )
    .select(
        "s.*",
        F.col("t.avg_goals_before_match").alias("away_avg_goals"),
        F.col("t.game_number").alias("away_game_number")
    )
)

result_df = away_joined_df.dropDuplicates()

# EDA Trend Analysis
------------------------------

In [0]:
### What is the distrbution of reuslts when the home team played more games than the away team
print('home_game_number > away_game_number')
display(result_df.where(F.col('home_game_number') > F.col('away_game_number')).groupBy('home_or_away_winner').count())

print('--------------------------------------')

### what is the distribution fo results when the away team played more games than the home team
print('home_game_number < away_game_number')
display(result_df.where(F.col('home_game_number') < F.col('away_game_number')).groupBy('home_or_away_winner').count())

### What is the distribution of results when the home team had a higher avg_goals_before_match than the away team
print('home_avg_goals > away_avg_goals')
display(result_df.where(F.col('home_avg_goals') > F.col('away_avg_goals')).groupBy('home_or_away_winner').count())

print('--------------------------------------')

### What is the distribution of results when the away team had a higher avg_goals_before_match than the home team
print('home_avg_goals < away_avg_goals')
display(result_df.where(F.col('home_avg_goals') < F.col('away_avg_goals')).groupBy('home_or_away_winner').count())

In [0]:
display(result_df)

In [0]:
result_df = (
    result_df
    .withColumn(
        "home_result_score",
        F.when(F.col("home_score") > F.col("away_score"), 1)
         .when(F.col("home_score") < F.col("away_score"), -1)
         .otherwise(0)
    )
    .withColumn(
        "away_result_score",
        F.when(F.col("away_score") > F.col("home_score"), 1)
         .when(F.col("away_score") < F.col("home_score"), -1)
         .otherwise(0)
    )
)

In [0]:
home_df = (
    result_df
    .select(
        "id",
        "date",
        F.col("home_team").alias("team"),
        F.col("home_result_score").alias("result_score")
    )
)

away_df = (
    result_df
    .select(
        "id",
        "date",
        F.col("away_team").alias("team"),
        F.col("away_result_score").alias("result_score")
    )
)

team_df = home_df.unionByName(away_df)


w_order = Window.partitionBy("team").orderBy("date")

w_1 = Window.partitionBy("team").orderBy("date").rowsBetween(-1, -1)

w_5 = Window.partitionBy("team").orderBy("date").rowsBetween(-5, -1)

w_10 = Window.partitionBy("team").orderBy("date").rowsBetween(-10, -1)

team_df = (
    team_df
    .withColumn("game_number", F.row_number().over(w_order))

    .withColumn(
        "form_last_1",
        F.when(
            F.col("game_number") >= 2,
            F.sum("result_score").over(w_1)
        )
    )

    .withColumn(
        "form_last_5",
        F.when(
            F.col("game_number") >= 6,
            F.sum("result_score").over(w_5)
        )
    )

    .withColumn(
        "form_last_10",
        F.when(
            F.col("game_number") >= 11,
            F.sum("result_score").over(w_10)
        )
    )
)

In [0]:
# Join form metrics back to result_df
team_form = team_df.select("id", "team", "form_last_1", "form_last_5", "form_last_10")

# Join for home team
result_df = (
    result_df.alias("r")
    .join(
        team_form.alias("h"),
        (F.col("r.id") == F.col("h.id")) & (F.col("r.home_team") == F.col("h.team")),
        "left"
    )
    .select(
        "r.*",
        F.col("h.form_last_1").alias("home_form_last_1"),
        F.col("h.form_last_5").alias("home_form_last_5"),
        F.col("h.form_last_10").alias("home_form_last_10")
    )
)

# Join for away team
result_df = (
    result_df.alias("r")
    .join(
        team_form.alias("a"),
        (F.col("r.id") == F.col("a.id")) & (F.col("r.away_team") == F.col("a.team")),
        "left"
    )
    .select(
        "r.*",
        F.col("a.form_last_1").alias("away_form_last_1"),
        F.col("a.form_last_5").alias("away_form_last_5"),
        F.col("a.form_last_10").alias("away_form_last_10")
    )
)

display(result_df)


#Write Gold layer data
--------------------------

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}")

In [0]:
result_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.{gold_table_name}")